In [1]:
import os 
import pandas as pd 
import geopandas as gpd
import logging 
import traceback
import numpy as np 
import requests
import json
from basicprocess import *
from shapely.geometry import Point, LineString, MultiLineString

In [2]:
'''Setup & Main Execution'''
# 00_Setup 所有全域函數
logfile = os.path.abspath(os.path.join(os.getcwd(), '..', 'Log', '台鐵分析.log'))
if os.path.exists(logfile):
    os.remove(logfile)
    
logging.basicConfig(
    filename=logfile,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)


In [3]:
# logging.info("Job Started")

# settup 
originalticket_folder = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '2024_2025'))
TRA_folder = os.path.join(originalticket_folder,'臺鐵電子票證資料(TO1A)') #臺鐵電子票證資料(TO1A)
reference_folder = os.path.join(os.getcwd(), '..', '參考資料')

# 初步篩選整理票證
TRA_primary_organized_folder = os.path.join(os.getcwd(), '..', '06_台鐵','01_初步篩選整理票證')
TRA_timeselect_folder = create_folder(os.path.join(TRA_primary_organized_folder, '01_指定時間區間票證'))
TRA_firstcount_folder = create_folder(os.path.join(TRA_primary_organized_folder, '02_上下車分票種分時計次'))


In [4]:
def TRA_tikcets_read_and_format(df, starttime, endtime, filterdate = ['2024-10-09', '2024-10-10', '2024-10-11', '2024-10-12', '2024-10-13', '2024-10-14', '2024-10-15']):
    df['Infodate'] = pd.to_datetime(df['Infodate'], format='%Y-%m-%d',errors = 'coerce')
    df['EntryTime'] = pd.to_datetime(df['EntryTime'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
    df['ExitTime'] = pd.to_datetime(df['ExitTime'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
    df['Hour'] = df['EntryTime'].dt.hour
    df['BordingHour'] = df['ExitTime'].dt.hour
    df['DeboardingHour'] = df['EntryTime'].dt.hour
    df['EntryStationID'] = df['EntryStationID'].astype('int64')
    df['ExitStationID'] = df['ExitStationID'].astype('int64')
    df['TimeSelect'] = (df['Infodate'] >= starttime) & (df['Infodate'] <= endtime)
    df['Error'] = df['EntryTime'] > df['ExitTime']


    df.loc[df['EntryStationID'] == 1001, 'EntryStationName'] = '臺北'
    df.loc[df['EntryStationID'] == 1001, 'EntryStationID'] = 1000

    df['DaysofWeek'] = df['Infodate'].dt.dayofweek
    wdwk_1_condition = df['DaysofWeek'].isin([1, 2, 3])
    # WDWK = -1 (週六=5, 週日=6)
    wdwk_neg1_condition = df['DaysofWeek'].isin([5, 6])
    # 使用 np.select (比多個 if/elif 判斷更快)
    df['WDWK'] = np.select(
        [wdwk_1_condition, wdwk_neg1_condition], # 條件列表
        [1, 0],                                # 對應的值
        default=-1                               # 預設值 (其他日子=1)
    )

    df['Month'] = df['Infodate'].dt.to_period('M')

    if filterdate and len(filterdate)>0:
        filterdate = ['2024-10-09', '2024-10-10', '2024-10-11', '2024-10-12', '2024-10-13', '2024-10-14', '2024-10-15']
        filter_dates_dt = pd.to_datetime(filterdate)
        df['FilterDate'] = df['Infodate'].isin(filter_dates_dt)

    return df

def step1_selectedtime(filepath, chunksize, starttime, endtime, outputfolder, filterdate):
    logging.info("開始讀取原始檔案並且篩選時間區間")
    logging.info(f"指定篩選時間為:{starttime}~{endtime}")

    outputpath = os.path.join(
        outputfolder,
        os.path.basename(filepath).replace('.csv', f'_{starttime}_to_{endtime}.csv')
    )
    outputpath2 = os.path.join(
        outputfolder,
        os.path.basename(filepath).replace('.csv', '_合理資料.csv')
    )

    if os.path.exists(outputpath):
        logging.warning(f'{os.path.basename(outputpath)} 已經存在；請再次檢查。')
    if os.path.exists(outputpath2):  # <- 你原本這裡也在檢查 outputpath（寫錯了）
        logging.warning(f'{os.path.basename(outputpath2)} 已經存在；請再次檢查。')
    chunks = pd.read_csv(filepath, skiprows=1, chunksize=chunksize)

    wrote_header_1 = False
    wrote_header_2 = False

    for chunk in chunks:
        chunk = TRA_tikcets_read_and_format(df=chunk, starttime=starttime, endtime=endtime, filterdate = filterdate)

        # --- 檔案 1：時間區間資料 ---
        chunk_time = chunk[chunk['TimeSelect']].drop(columns=['TimeSelect'])
        if len(chunk_time) > 0:
            chunk_time.to_csv(
                outputpath,
                mode='w' if not wrote_header_1 else 'a',
                header=not wrote_header_1,
                index=False,
                encoding='utf-8-sig'
            )
            wrote_header_1 = True

        # --- 檔案 2：合理資料（Error=False）---
        chunk_ok = chunk_time[chunk_time['Error'] == False].drop(columns=['Error'])
        if len(chunk_ok) > 0:
            chunk_ok.to_csv(
                outputpath2,
                mode='w' if not wrote_header_2 else 'a',
                header=not wrote_header_2,
                index=False,
                encoding='utf-8-sig'
            )
            wrote_header_2 = True

    logging.info(f"篩選出時間為 {starttime}~{endtime} 的資料")

def step2_wdwk_count(filepath, outputfolder, filterdate):
    logging.info('Job_step2_wdwk_count Started')
    logging.info('開始統計票種各站點、小時進出次數')

    outputpath = os.path.join(outputfolder, os.path.basename(filepath).replace('_合理資料.csv', '_分票種及平假日統計.csv'))

    df = pd.read_csv(filepath)

    if filterdate :
        df = df[df['FilterDate'] == False]

    groupbycolumns =[ 'Month', 'Hour', 'WDWK', 'HolderType', 'BordingHour', 'EntryStationID', 'EntryStationName', 'ExitStationID', 'ExitStationName',   'DeboardingHour']
    df = df.reindex(columns=groupbycolumns)
    df[groupbycolumns] = df[groupbycolumns].fillna(-999)
    df = df.groupby(groupbycolumns).size().reset_index(name='Count')

    df.to_csv(outputpath, index=False)

    logging.info('Job_step2_wdwk_count Finished')

def get_stacode(downloadfolder):

    StaCode_URL = 'https://ods.railway.gov.tw/tra-ods-web/ods/download/dataResource/0518b833e8964d53bfea3f7691aea0ee'
    response = requests.get(StaCode_URL)
    # 保存 JSON 文件到指定的資料夾
    # logging.info("下載臺鐵站點資料")
    try:
        json_file_path = os.path.join(downloadfolder, 'station_data.json')
        with open(json_file_path, 'wb') as f:
            f.write(response.content)
        # 打開並讀取 JSON 文件
        with open(json_file_path, 'r', encoding='utf-8') as f:
            station_data = f.read()
        logging.info(f'台鐵站點資料下載成功 : {json_file_path}')
        StaCode = json.loads(station_data)
        StaCode = pd.json_normalize(StaCode)
        StaCode['stationCode'] = pd.to_numeric(StaCode['stationCode'], errors = 'coerce')
        StaCode['stationCode'] = StaCode['stationCode'].astype('int64')
        coords = StaCode['gps'].astype(str).str.strip().str.split()
        StaCode['Lat'] = pd.to_numeric(coords.str[0], errors='coerce')
        StaCode['Lon'] = pd.to_numeric(coords.str[1], errors='coerce')
        StaCode = StaCode.reindex(columns= ['stationCode', 'stationName', 'stationEName', 'stationAddrTw', 'Lat', 'Lon'])
        return StaCode
    except:
        logging.error('取得完整台鐵資料失敗')

def dataframe_to_point(df, lon_col, lat_col, crs="EPSG:4326", target_crs="EPSG:3826"):
    '''
    Parameters:
    df (dataframe) : 含經緯度座標欄位的dataframe
    lon_col (str) : 緯度欄位
    Lat_col (str) : 經度欄位
    crs (str) : 目前經緯度座標的座標系統，常用的為4326(WGS84)、3826(TWD97)
    target_crs：目標轉換的座標系統
    '''

    # from shapely.geometry import Point
    # import pandas as pd
    # import geopandas as gpd
    # Create Point geometries from the longitude and latitude columns
    geometry = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    # Create a GeoDataFrame with the original CRS
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)
    # Convert the GeoDataFrame to the target CRS
    gdf = gdf.to_crs(epsg=target_crs.split(":")[1])
    return gdf

def get_station_with_place(downloadfolder, reference_folder):

    station = get_stacode(downloadfolder = downloadfolder)
    placeshape = gpd.read_file(os.path.join(reference_folder, '縣市界.shp'))
    placeshape = placeshape.reindex(columns = ['COUNTYID', 'COUNTYNAME', 'geometry'])
    station =  dataframe_to_point(df = station, lon_col = 'Lon', lat_col = 'Lat', crs="EPSG:4326", target_crs="EPSG:3826")
    station = station.reindex(columns = ['stationCode', 'stationName', 'Lat',  'Lon', 'geometry'])

    station = gpd.sjoin(
                        station,
                        placeshape,
                        how="left",
                        predicate="within"   # 若點剛好壓在邊界上，可能會變成 NaN；可改 intersects
                        ).drop(columns=["index_right"])
    
    return station.drop(columns = 'geometry')

def step3_od_count_with_county(reference_folder, TRA_firstcount_folder):

    station = get_station_with_place(downloadfolder = reference_folder, reference_folder = reference_folder)
    station = station.drop(columns = ['stationName'])
    station.drop_duplicates(subset = ['stationCode'])

    file = [f for f in findfiles(TRA_firstcount_folder) if '分票種及平假日統計' in f][0]

    df = pd.read_csv(file)
    datanumbers = int(df['Count'].sum())

    df = df.merge(station, left_on=['EntryStationID'], right_on=['stationCode'])
    df = df.merge(station, left_on=['ExitStationID'], right_on=['stationCode'], suffixes=['_O', '_D'])

    tmp = int(df['Count'].sum())
    if tmp > datanumbers:
        logging.error(f"黏貼站點縣市後出現問題，資料筆數從 {datanumbers:,}筆增加為{tmp:,}筆")
    del tmp 


    df_date = pd.read_excel(os.path.join(reference_folder, 'Date.xlsx'), sheet_name = 'DateCount')
    df_date['Month'] = pd.to_datetime(df_date['Month']).dt.to_period('M')

    reindex_columns = ['Month', 'Hour', 'WDWK', 'HolderType', 
                       'BordingHour', 'EntryStationID','EntryStationName',  'Lat_O', 'Lon_O', 'COUNTYID_O', 'COUNTYNAME_O',
                       'DeboardingHour', 'ExitStationID', 'ExitStationName', 'Lat_D', 'Lon_D', 'COUNTYID_D', 'COUNTYNAME_D', 
                       'Count']
    df = df.reindex(columns=reindex_columns)
    outputpath = os.path.join(TRA_firstcount_folder, os.path.basename(file).replace('_分票種及平假日統計.csv', '_分票種平假日及所在縣市.csv'))
    df.to_csv(outputpath, index=False)
    logging.info(f'輸出含縣市的OD統計資料:{outputpath}')

    df['Month'] = pd.to_datetime(df['Month']).dt.to_period('M')

    on_groupbycolumns = ['Month', 'WDWK', 'BordingHour', 'EntryStationID', 'COUNTYID_O', 'COUNTYNAME_O']
    df_on = df.groupby(on_groupbycolumns).agg({'Count':'sum'}).reset_index()
    # df_on['Month'] = pd.to_datetime(df_on['Month']).dt.to_period('M')
    df_on['Month'] = df_on['Month'].astype('period[M]')
    df_on = df_on.merge(df_date, on = ['Month', 'WDWK'])
    df_on['Avg'] = df_on['Count']/df_on['DayCount']
    outputpath = os.path.join(TRA_firstcount_folder, os.path.basename(file).replace('_分票種及平假日統計.csv', '_分時上車站點統計.csv'))
    df_on.to_csv(outputpath, index=False)


    on_groupbycolumns = ['Month', 'WDWK', 'DeboardingHour', 'ExitStationID', 'COUNTYID_D', 'COUNTYNAME_D']
    df_off = df.groupby(on_groupbycolumns).agg({'Count':'sum'}).reset_index()
    # df_off['Month'] = pd.to_datetime(df_off['Month']).dt.to_period('M')
    df_off['Month'] = df_off['Month'].astype('period[M]')
    df_off = df_off.merge(df_date, on = ['Month', 'WDWK'])
    df_off['Avg'] = df_off['Count']/df_off['DayCount']
    outputpath = os.path.join(TRA_firstcount_folder, os.path.basename(file).replace('_分票種及平假日統計.csv', '_分時下車站點統計.csv'))
    df_off.to_csv(outputpath, index=False)

    combinedcolumns = ['Month','WDWK','Hour','StationID','COUNTYID_O', 'COUNTYNAME_O','Count','DayCount','Avg']
    df_on.columns = combinedcolumns
    df_off.columns = combinedcolumns
    df_on['ONF'] = 'On'
    df_off['ONF'] = 'Off'
    df_combined = pd.concat([df_on, df_off])
    df_combined.to_csv(outputpath.replace('_分時下車站點統計.csv', '_分時上下車站點統計.csv'))

def main():
    # logging.info('Job Started')

    # 讀取台鐵票證資料 篩選正確區間
    # step1_selectedtime(filepath = findfiles(TRA_folder)[0], 
    #                    chunksize = 1000000, 
    #                    starttime = '2024-10-01', 
    #                    endtime = '2024-11-30', 
    #                    outputfolder = TRA_timeselect_folder, 
    #                    filterdate = ['2024-10-09', '2024-10-10', '2024-10-11', '2024-10-12', '2024-10-13', '2024-10-14', '2024-10-15'])

    # step2_wdwk_count(filepath =  [f for f in findfiles(TRA_timeselect_folder) if '合理資料' in f][0], 
    #                  outputfolder = TRA_firstcount_folder, 
    #                  filterdate = True)
    
    step3_od_count_with_county(reference_folder, TRA_firstcount_folder)
    

In [5]:
if __name__ == "__main__":
    try:
        main()

    except Exception as e:
        logging.error("main() 執行失敗：%s", e)  
        logging.error("Traceback:\n%s", traceback.format_exc())
    
    outputlog(logfile=logfile)